In [1]:
import torch
import torch.nn as nn
from torchinfo import summary
from torch.utils.data import DataLoader,Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using {device}")

using cuda


In [3]:
df_train=pd.read_csv(r'C:\Coding\ML_DL\Datasets\fashion-mnist_train.csv')
df_test=pd.read_csv(r'C:\Coding\ML_DL\Datasets\fashion-mnist_test.csv')


In [4]:
X_train=df_train.iloc[:,1:]
X_test=df_test.iloc[:,1:]
y_train=df_train.iloc[:,0]
y_test=df_test.iloc[:,0]


X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_test  = X_test.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

X_train=X_train/255
X_test=X_test/255

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32).reshape(-1,1,28,28)      #(B,C,H,,W)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    

train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,pin_memory=True)


In [21]:
class ConvNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        self.feature_extraction=nn.Sequential(
            nn.Conv2d(input_features,out_channels=32,kernel_size=3,padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear((64*7*7),128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,10)

        )
        

    def forward(self,X):
        X = self.feature_extraction(X)
        return self.classifier(X)
        

In [22]:
learning_rate=0.01
epochs=100

In [23]:
model=ConvNN(1)              #no of input channels
model.to(device)

optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)
criterion=nn.CrossEntropyLoss()

In [24]:
for epoch in range(epochs):
    total_epoch_loss=0
    for batch_features,batch_labels in train_loader:
        batch_features=batch_features.to(device)
        batch_labels=batch_labels.to(device)
        #forward
        output=model(batch_features)
        #loss 
        loss=criterion(output,batch_labels)
        #back
        optimizer.zero_grad()
        loss.backward()
        #update params
        optimizer.step()
        total_epoch_loss+= loss.item()

    avg_loss=total_epoch_loss/len(train_loader)
    if(epoch%10==0):
        print(f'Epoch {epoch+1}: Loss{avg_loss}')


Epoch 1: Loss0.6209678676446279
Epoch 11: Loss0.19008204874296983
Epoch 21: Loss0.12095895038346449
Epoch 31: Loss0.07478992759097988
Epoch 41: Loss0.04713308033412322
Epoch 51: Loss0.031977281015428405
Epoch 61: Loss0.0227085638969574
Epoch 71: Loss0.017455777993289907
Epoch 81: Loss0.014602451341284904
Epoch 91: Loss0.011998041588818887


In [25]:
model.eval()
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.9229


# using vgg16

In [45]:
from torchvision.transforms import transforms
import torchvision.models as models
from PIL import Image
import numpy as np

custom_transform=transforms.Compose([transforms.Resize(256),
                                     transforms.CenterCrop(224),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])

class TrfLrnDataset(Dataset):
    def __init__(self, features, labels, transform=None):
        self.features = features.values   # keep as NumPy
        self.labels = labels.values
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # get image (28x28)
        image = self.features[idx].reshape(28, 28)

        # convert to uint8
        image = image.astype(np.uint8)

        # grayscale → RGB (H, W, 3)
        image = np.stack([image] * 3, axis=-1)

        # NumPy → PIL
        image = Image.fromarray(image)

        # apply transforms
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return image, label

    

train_dataset=TrfLrnDataset(X_train,y_train,transform=custom_transform)
test_dataset=TrfLrnDataset(X_test,y_test,transform=custom_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,pin_memory=True)



In [33]:
model=models.vgg16(pretrained=True)

c:\Coding\ML_DL\deep-learning\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Coding\ML_DL\deep-learning\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\narin/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [00:49<00:00, 11.2MB/s] 


In [34]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [40]:
for param in model.features.parameters():
    param.requires_grad=False #freeze training

model.classifier=nn.Sequential(
            nn.Linear(25088,1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024,512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512,10)
    
)


In [41]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=1024, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=1024, out_features=512, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=512, out_features=10, bias=True)
)

In [42]:
vgg16=model.to(device)

In [43]:
optimizer=optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)

In [46]:
for epoch in range(10):
    total_epoch_loss=0
    for batch_features,batch_labels in train_loader:
        batch_features=batch_features.to(device)
        batch_labels=batch_labels.to(device)
        #forward
        output=vgg16(batch_features)
        #loss 
        loss=criterion(output,batch_labels)
        #back
        optimizer.zero_grad()
        loss.backward()
        #update params
        optimizer.step()
        total_epoch_loss+= loss.item()

    avg_loss=total_epoch_loss/len(train_loader)
    if(epoch%10==0):
        print(f'Epoch {epoch}: Loss {avg_loss}')


Epoch 0: Loss2.3464137348175047


In [47]:
model.eval()
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.1
